# 🐍 Day 31 — Design Patterns
### Python Advanced Mastery · Days 31–57

---

> **"Design patterns are reusable solutions to commonly occurring problems in software design."**  
> — Gang of Four (GoF)

## 📋 What You'll Learn Today

| Category | Patterns Covered |
|----------|------------------|
| **Creational** | Singleton, Factory, Builder |
| **Structural** | Adapter, Decorator, Proxy |
| **Behavioral** | Observer, Strategy, Command |
| **Python-Specific** | Pythonic rewrites of classic patterns |
| **Anti-Patterns** | God Object, Spaghetti, Magic Numbers |

---

## 🗺️ Notebook Structure

1. [Creational Patterns](#creational)
2. [Structural Patterns](#structural)
3. [Behavioral Patterns](#behavioral)
4. [Python-Specific Patterns](#pythonic)
5. [Anti-Patterns](#antipatterns)
6. [Quiz + Answers](#quiz)
7. [Interview Questions + Answers](#interview)
8. [Problem Statement + Solution](#problem)
9. [Real-World Use-Case: Event-Driven Notification System](#usecase)

---
## 1. Creational Patterns <a id='creational'></a>

Creational patterns deal with **object creation mechanisms**. They decouple the instantiation process from the rest of the code.

---
### 1.1 Singleton Pattern

**Intent:** Ensure a class has **only one instance** and provide a global access point.

**When to use:**
- Database connection pools
- Logger instances
- Configuration managers
- Thread pools

In [ ]:
# ── Approach 1: Using __new__ ──────────────────────────────────────────────────
class SingletonMeta(type):
    """Thread-safe Singleton using a metaclass."""
    _instances: dict = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self, url: str = "sqlite:///app.db"):
        self.url = url
        self.connected = False
        print(f"[DB] Initialised connection to {url}")

    def connect(self):
        self.connected = True
        return self

    def __repr__(self):
        return f"DatabaseConnection(url={self.url!r}, connected={self.connected})"


# ── Demo ──────────────────────────────────────────────────────────────────────
db1 = DatabaseConnection("postgresql://localhost/mydb")
db2 = DatabaseConnection()  # __init__ is NOT called again — same instance

print(f"db1 is db2: {db1 is db2}")          # True
print(f"id(db1) == id(db2): {id(db1) == id(db2)}")  # True
print(db1)

In [ ]:
# ── Approach 2: Pythonic — module-level singleton ─────────────────────────────
# In Python, a module is already a singleton — imported once and cached.
# The cleanest Pythonic singleton is just a module-level instance.

class _AppConfig:
    """Private class — only one instance should exist."""
    def __init__(self):
        self.debug = False
        self.version = "1.0.0"
        self.max_workers = 4

    def update(self, **kwargs):
        for key, val in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, val)
        return self

    def __repr__(self):
        return f"AppConfig(debug={self.debug}, version={self.version!r}, workers={self.max_workers})"


# Module-level instance — this IS the singleton
config = _AppConfig()

config.update(debug=True, version="2.0.0")
print(config)

---
### 1.2 Factory Pattern

**Intent:** Define an interface for creating objects, but let subclasses (or a factory function) decide which class to instantiate.

**When to use:**
- When the exact type of object is determined at runtime
- When you want to centralise object creation logic
- Plugin systems, parsers, serialisers

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Literal


# ── Product hierarchy ─────────────────────────────────────────────────────────
class Notification(ABC):
    """Abstract product."""
    @abstractmethod
    def send(self, message: str, recipient: str) -> str: ...


class EmailNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"[EMAIL] To: {recipient} | {message}"


class SMSNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"[SMS]   To: {recipient} | {message[:160]}"


class PushNotification(Notification):
    def send(self, message: str, recipient: str) -> str:
        return f"[PUSH]  To: {recipient} | {message[:50]}"


# ── Factory ───────────────────────────────────────────────────────────────────
NotificationType = Literal["email", "sms", "push"]

_REGISTRY: dict[str, type[Notification]] = {
    "email": EmailNotification,
    "sms":   SMSNotification,
    "push":  PushNotification,
}

def notification_factory(kind: NotificationType) -> Notification:
    """Factory function — returns the right Notification object."""
    cls = _REGISTRY.get(kind)
    if cls is None:
        raise ValueError(f"Unknown notification type: {kind!r}. Choose from {list(_REGISTRY)}")
    return cls()


# ── Demo ──────────────────────────────────────────────────────────────────────
for channel in ["email", "sms", "push"]:
    notif = notification_factory(channel)
    print(notif.send("Your task 'Write report' is due tomorrow!", "alice@example.com"))

---
### 1.3 Builder Pattern

**Intent:** Separate the **construction** of a complex object from its **representation**. Use the same construction process to create different representations.

**When to use:**
- Objects with many optional parameters (avoid telescoping constructors)
- Step-by-step construction of complex objects
- SQL query builders, HTTP request builders

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class QueryConfig:
    table: str
    columns: list[str] = field(default_factory=lambda: ["*"])
    conditions: list[str] = field(default_factory=list)
    order_by: Optional[str] = None
    limit: Optional[int] = None
    offset: int = 0


class QueryBuilder:
    """Fluent Builder for SQL SELECT queries."""

    def __init__(self, table: str):
        self._config = QueryConfig(table=table)

    def select(self, *columns: str) -> QueryBuilder:
        self._config.columns = list(columns)
        return self   # ← return self enables method chaining

    def where(self, condition: str) -> QueryBuilder:
        self._config.conditions.append(condition)
        return self

    def order_by(self, column: str, direction: str = "ASC") -> QueryBuilder:
        self._config.order_by = f"{column} {direction}"
        return self

    def limit(self, n: int) -> QueryBuilder:
        self._config.limit = n
        return self

    def offset(self, n: int) -> QueryBuilder:
        self._config.offset = n
        return self

    def build(self) -> str:
        c = self._config
        cols = ", ".join(c.columns)
        sql = f"SELECT {cols} FROM {c.table}"
        if c.conditions:
            sql += " WHERE " + " AND ".join(c.conditions)
        if c.order_by:
            sql += f" ORDER BY {c.order_by}"
        if c.limit:
            sql += f" LIMIT {c.limit}"
        if c.offset:
            sql += f" OFFSET {c.offset}"
        return sql + ";"


# ── Demo ──────────────────────────────────────────────────────────────────────
query = (
    QueryBuilder("tasks")
    .select("id", "title", "due_date", "status")
    .where("status != 'done'")
    .where("due_date < NOW()")
    .order_by("due_date", "ASC")
    .limit(20)
    .offset(40)
    .build()
)
print(query)

---
## 2. Structural Patterns <a id='structural'></a>

Structural patterns deal with **object composition** — how classes and objects are assembled into larger structures.

---
### 2.1 Adapter Pattern

**Intent:** Convert the interface of a class into another interface clients expect. Allows incompatible interfaces to work together.

**Analogy:** A power plug adapter — same electricity, different socket shape.

In [ ]:
# ── Scenario: Our app expects a `Logger` interface, but we're using a
#   third-party library with a completely different interface.

# ── Target interface (what our app uses) ──────────────────────────────────────
class Logger(ABC):
    @abstractmethod
    def log(self, level: str, message: str) -> None: ...

    @abstractmethod
    def close(self) -> None: ...


# ── Adaptee: legacy/third-party logger (incompatible interface) ───────────────
class LegacyFileLogger:
    """Pretend this comes from a third-party library we can't modify."""
    def open_file(self, path: str) -> None:
        print(f"[Legacy] Opened log file: {path}")

    def write_to_file(self, text: str) -> None:
        print(f"[Legacy] Writing: {text}")

    def close_file(self) -> None:
        print("[Legacy] Closed log file.")


# ── Adapter ───────────────────────────────────────────────────────────────────
class LegacyLoggerAdapter(Logger):
    """Wraps LegacyFileLogger to satisfy our Logger interface."""

    def __init__(self, path: str = "/var/log/app.log"):
        self._legacy = LegacyFileLogger()
        self._legacy.open_file(path)

    def log(self, level: str, message: str) -> None:
        # Translate our interface call → legacy interface call
        self._legacy.write_to_file(f"[{level.upper()}] {message}")

    def close(self) -> None:
        self._legacy.close_file()


# ── App code only knows about Logger — not LegacyFileLogger ──────────────────
def run_app(logger: Logger):
    logger.log("info",  "Application started")
    logger.log("warn",  "Config file not found, using defaults")
    logger.log("error", "Database timeout after 30s")
    logger.close()


run_app(LegacyLoggerAdapter())

---
### 2.2 Decorator Pattern

**Intent:** Attach additional responsibilities to an object **dynamically**. Decorators provide a flexible alternative to subclassing.

> ⚡ Python's `@decorator` syntax is a first-class language feature that directly implements this pattern!

In [ ]:
import time
import functools
from typing import Callable, TypeVar, Any

F = TypeVar("F", bound=Callable[..., Any])


# ── Decorator 1: Timing ───────────────────────────────────────────────────────
def timer(func: F) -> F:
    """Measure and print execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper  # type: ignore


# ── Decorator 2: Retry with exponential backoff ───────────────────────────────
def retry(max_attempts: int = 3, delay: float = 0.5, exceptions: tuple = (Exception,)):
    """Retry a function on failure with exponential backoff."""
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    wait = delay * (2 ** (attempt - 1))
                    print(f"[retry] Attempt {attempt} failed ({e}). Retrying in {wait:.1f}s...")
                    time.sleep(wait)
        return wrapper  # type: ignore
    return decorator


# ── Decorator 3: Cache (manual memoisation) ───────────────────────────────────
def memoize(func: F) -> F:
    """Simple memoisation cache."""
    cache: dict = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper  # type: ignore


# ── Stacking decorators ───────────────────────────────────────────────────────
@timer
@memoize
def fibonacci(n: int) -> int:
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print(f"fib(35) = {fibonacci(35)}")
print(f"fib(35) = {fibonacci(35)}  ← from cache")

# ── Retry demo ────────────────────────────────────────────────────────────────
_call_count = 0

@retry(max_attempts=3, delay=0.1, exceptions=(ConnectionError,))
def flaky_api_call() -> str:
    global _call_count
    _call_count += 1
    if _call_count < 3:
        raise ConnectionError("Service unavailable")
    return "✓ API response received"

print(flaky_api_call())

---
### 2.3 Proxy Pattern

**Intent:** Provide a **surrogate or placeholder** for another object to control access to it.

**Types:**
- **Virtual proxy** — lazy initialisation
- **Protection proxy** — access control
- **Caching proxy** — cache results
- **Remote proxy** — represent a remote object

In [ ]:
from typing import Optional


# ── Subject interface ─────────────────────────────────────────────────────────
class DataService(ABC):
    @abstractmethod
    def fetch(self, key: str) -> dict: ...


# ── Real subject (expensive) ──────────────────────────────────────────────────
class RemoteDataService(DataService):
    def fetch(self, key: str) -> dict:
        print(f"[RemoteDataService] Making expensive network call for {key!r}...")
        time.sleep(0.05)  # simulate latency
        return {"key": key, "value": f"data_for_{key}", "ttl": 300}


# ── Caching proxy ─────────────────────────────────────────────────────────────
class CachingProxy(DataService):
    """Transparent caching proxy — client doesn't know about the cache."""

    def __init__(self, service: DataService):
        self._service = service
        self._cache: dict[str, dict] = {}
        self._hits = 0
        self._misses = 0

    def fetch(self, key: str) -> dict:
        if key not in self._cache:
            self._misses += 1
            self._cache[key] = self._service.fetch(key)
        else:
            self._hits += 1
            print(f"[CachingProxy] Cache HIT for {key!r}")
        return self._cache[key]

    @property
    def stats(self) -> str:
        total = self._hits + self._misses
        ratio = self._hits / total * 100 if total else 0
        return f"Cache: {self._hits} hits / {self._misses} misses ({ratio:.0f}% hit rate)"


# ── Demo ──────────────────────────────────────────────────────────────────────
service = CachingProxy(RemoteDataService())

for key in ["user:1", "user:2", "user:1", "user:1", "user:3", "user:2"]:
    result = service.fetch(key)

print()
print(service.stats)

---
## 3. Behavioral Patterns <a id='behavioral'></a>

Behavioral patterns deal with **communication between objects** — how they interact and distribute responsibility.

---
### 3.1 Observer Pattern

**Intent:** Define a **one-to-many dependency** so that when one object changes state, all its dependents are notified automatically.

**Also known as:** Publish-Subscribe, Event System, Listener pattern

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable
from collections import defaultdict


# ── Observer protocol ─────────────────────────────────────────────────────────
@runtime_checkable
class Observer(Protocol):
    def update(self, event: str, data: dict) -> None: ...


# ── Event Bus (Subject) ───────────────────────────────────────────────────────
class EventBus:
    """A simple publish-subscribe event bus."""

    def __init__(self):
        self._listeners: dict[str, list[Observer]] = defaultdict(list)

    def subscribe(self, event: str, observer: Observer) -> None:
        self._listeners[event].append(observer)
        print(f"[EventBus] {observer.__class__.__name__} subscribed to '{event}'")

    def unsubscribe(self, event: str, observer: Observer) -> None:
        self._listeners[event].remove(observer)

    def publish(self, event: str, data: dict) -> None:
        print(f"\n[EventBus] Publishing '{event}' → {len(self._listeners[event])} listener(s)")
        for observer in self._listeners[event]:
            observer.update(event, data)


# ── Concrete Observers ────────────────────────────────────────────────────────
class AuditLogger:
    def update(self, event: str, data: dict) -> None:
        print(f"  [AuditLogger]   LOGGED  → event={event!r}, data={data}")


class EmailAlerter:
    def update(self, event: str, data: dict) -> None:
        user = data.get("user", "unknown")
        print(f"  [EmailAlerter]  SENT    → email to {user} about '{event}'")


class MetricsDashboard:
    def __init__(self):
        self._counts: dict[str, int] = defaultdict(int)

    def update(self, event: str, data: dict) -> None:
        self._counts[event] += 1
        print(f"  [MetricsDash]   COUNT   → '{event}' occurred {self._counts[event]} time(s)")


# ── Demo ──────────────────────────────────────────────────────────────────────
bus = EventBus()
logger    = AuditLogger()
emailer   = EmailAlerter()
dashboard = MetricsDashboard()

bus.subscribe("task.created",   logger)
bus.subscribe("task.created",   emailer)
bus.subscribe("task.created",   dashboard)
bus.subscribe("task.completed", logger)
bus.subscribe("task.completed", dashboard)

bus.publish("task.created",   {"task_id": 42, "title": "Write tests",   "user": "alice@example.com"})
bus.publish("task.created",   {"task_id": 43, "title": "Deploy to prod", "user": "bob@example.com"})
bus.publish("task.completed", {"task_id": 42, "title": "Write tests",   "user": "alice@example.com"})

---
### 3.2 Strategy Pattern

**Intent:** Define a family of algorithms, encapsulate each one, and make them interchangeable. Strategy lets the algorithm vary independently from clients that use it.

In [ ]:
from typing import Callable

# ── Pythonic Strategy: use plain callables / Protocol ─────────────────────────
# Instead of abstract base classes, Python can use callables directly.

SortStrategy = Callable[[list], list]


def bubble_sort(data: list) -> list:
    arr = data.copy()
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


def merge_sort(data: list) -> list:
    if len(data) <= 1:
        return data
    mid = len(data) // 2
    left  = merge_sort(data[:mid])
    right = merge_sort(data[mid:])
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]);  i += 1
        else:                    result.append(right[j]); j += 1
    return result + left[i:] + right[j:]


def python_builtin_sort(data: list) -> list:
    return sorted(data)


# ── Context ───────────────────────────────────────────────────────────────────
class Sorter:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    def set_strategy(self, strategy: SortStrategy) -> None:
        """Swap strategy at runtime."""
        self._strategy = strategy

    def sort(self, data: list) -> list:
        start = time.perf_counter()
        result = self._strategy(data)
        elapsed = time.perf_counter() - start
        print(f"[{self._strategy.__name__:<22}] sorted {len(data):>5} items in {elapsed*1000:.3f}ms")
        return result


import random
data = [random.randint(0, 1000) for _ in range(500)]

sorter = Sorter(bubble_sort)
sorter.sort(data)

sorter.set_strategy(merge_sort)
sorter.sort(data)

sorter.set_strategy(python_builtin_sort)
sorter.sort(data)

---
### 3.3 Command Pattern

**Intent:** Encapsulate a request as an object, thereby letting you parameterise clients with different requests, queue or log requests, and support undoable operations.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
import copy


# ── Command interface ─────────────────────────────────────────────────────────
class Command(ABC):
    @abstractmethod
    def execute(self) -> str: ...

    @abstractmethod
    def undo(self) -> str: ...


# ── Receiver ──────────────────────────────────────────────────────────────────
@dataclass
class TextDocument:
    content: str = ""

    def insert(self, position: int, text: str) -> None:
        self.content = self.content[:position] + text + self.content[position:]

    def delete(self, start: int, length: int) -> str:
        deleted = self.content[start:start + length]
        self.content = self.content[:start] + self.content[start + length:]
        return deleted


# ── Concrete commands ─────────────────────────────────────────────────────────
class InsertCommand(Command):
    def __init__(self, doc: TextDocument, position: int, text: str):
        self._doc = doc
        self._position = position
        self._text = text

    def execute(self) -> str:
        self._doc.insert(self._position, self._text)
        return f"Inserted {self._text!r} at position {self._position}"

    def undo(self) -> str:
        self._doc.delete(self._position, len(self._text))
        return f"Undone: removed {self._text!r} from position {self._position}"


# ── Invoker (Command History / Undo stack) ────────────────────────────────────
class CommandHistory:
    def __init__(self):
        self._history: list[Command] = []
        self._undone:  list[Command] = []

    def execute(self, command: Command) -> None:
        result = command.execute()
        self._history.append(command)
        self._undone.clear()  # clear redo stack on new action
        print(f"  ✓ {result}")

    def undo(self) -> None:
        if not self._history:
            print("  Nothing to undo.")
            return
        cmd = self._history.pop()
        result = cmd.undo()
        self._undone.append(cmd)
        print(f"  ↩ {result}")

    def redo(self) -> None:
        if not self._undone:
            print("  Nothing to redo.")
            return
        cmd = self._undone.pop()
        result = cmd.execute()
        self._history.append(cmd)
        print(f"  ↪ {result}")


# ── Demo ──────────────────────────────────────────────────────────────────────
doc     = TextDocument("Hello World")
history = CommandHistory()

print(f"Initial: {doc.content!r}")
history.execute(InsertCommand(doc, 5, ", Python"))
print(f"After insert:  {doc.content!r}")

history.execute(InsertCommand(doc, len(doc.content), "!"))
print(f"After insert:  {doc.content!r}")

history.undo()
print(f"After undo:    {doc.content!r}")

history.redo()
print(f"After redo:    {doc.content!r}")

---
## 4. Python-Specific Patterns <a id='pythonic'></a>

Classic GoF patterns were designed for statically-typed OO languages (Java/C++). Python's dynamic nature, first-class functions, and rich built-ins enable cleaner, more Pythonic implementations.

---
### 4.1 Registry Pattern (Plugin System)

In [ ]:
# ── Registry via class decorator ──────────────────────────────────────────────
from typing import TypeVar, Callable

HandlerFunc = Callable[[dict], dict]


class HandlerRegistry:
    """Self-registering handler registry — plugins register themselves."""

    def __init__(self):
        self._handlers: dict[str, HandlerFunc] = {}

    def register(self, event_type: str):
        """Decorator — registers a function as handler for event_type."""
        def decorator(func: HandlerFunc) -> HandlerFunc:
            self._handlers[event_type] = func
            print(f"[Registry] Registered handler '{func.__name__}' for '{event_type}'")
            return func
        return decorator

    def dispatch(self, event_type: str, payload: dict) -> dict:
        handler = self._handlers.get(event_type)
        if handler is None:
            raise KeyError(f"No handler for event type: {event_type!r}")
        return handler(payload)

    def registered_events(self) -> list[str]:
        return list(self._handlers.keys())


# ── Usage: handlers register themselves ───────────────────────────────────────
registry = HandlerRegistry()

@registry.register("user.signup")
def handle_user_signup(payload: dict) -> dict:
    return {"action": "send_welcome_email", "to": payload["email"]}

@registry.register("order.placed")
def handle_order_placed(payload: dict) -> dict:
    return {"action": "notify_warehouse", "order_id": payload["order_id"]}

@registry.register("task.overdue")
def handle_task_overdue(payload: dict) -> dict:
    return {"action": "send_reminder", "task": payload["title"]}


print(f"\nRegistered events: {registry.registered_events()}")
print(registry.dispatch("task.overdue", {"title": "Submit expense report", "user": "bob"}))

---
### 4.2 Context Manager Pattern (`__enter__` / `__exit__`)

In [ ]:
import contextlib
import time


# ── Class-based context manager ───────────────────────────────────────────────
class ManagedTransaction:
    """Fake database transaction context manager."""

    def __init__(self, name: str):
        self.name = name
        self._committed = False

    def __enter__(self):
        print(f"[TX:{self.name}] BEGIN")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            print(f"[TX:{self.name}] COMMIT")
            self._committed = True
        else:
            print(f"[TX:{self.name}] ROLLBACK — {exc_type.__name__}: {exc_val}")
        return False  # Don't suppress exceptions

    def execute(self, sql: str) -> None:
        print(f"[TX:{self.name}] EXECUTE: {sql}")


# ── Generator-based context manager with @contextmanager ─────────────────────
@contextlib.contextmanager
def timer_ctx(label: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[timer] {label}: {elapsed*1000:.2f}ms")


# ── Demo ──────────────────────────────────────────────────────────────────────
with ManagedTransaction("create_task") as tx:
    tx.execute("INSERT INTO tasks (title) VALUES ('Deploy v2.0')")
    tx.execute("INSERT INTO audit_log (action) VALUES ('task_created')")

print()

try:
    with ManagedTransaction("bad_update") as tx:
        tx.execute("UPDATE tasks SET owner_id = 999")
        raise ValueError("Foreign key constraint violated")
except ValueError:
    pass

print()
with timer_ctx("sorting 100k items"):
    sorted(range(100_000, 0, -1))

---
## 5. Anti-Patterns <a id='antipatterns'></a>

**Anti-patterns** are common solutions that seem reasonable but make code harder to maintain, test, or understand.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Anti-Pattern 1: GOD OBJECT — knows and does too much
# ═══════════════════════════════════════════════════════════════

# ❌ BAD — God Object
class App:
    def handle_user(self, user): pass
    def send_email(self, to, msg): pass
    def query_db(self, sql): pass
    def render_html(self, template): pass
    def write_log(self, msg): pass
    def process_payment(self, amount): pass
    def generate_pdf(self, data): pass

# ✅ GOOD — Single Responsibility, separate classes
class UserService: ...
class EmailService: ...
class Database: ...
class TemplateRenderer: ...

print("Anti-Pattern 1: God Object → Split by Single Responsibility Principle")


# ═══════════════════════════════════════════════════════════════
# Anti-Pattern 2: MAGIC NUMBERS / STRINGS
# ═══════════════════════════════════════════════════════════════
from enum import Enum, auto

# ❌ BAD
def process_task_bad(task):
    if task["status"] == 2:      # what is 2?
        if task["priority"] > 7: # why 7?
            return True

# ✅ GOOD
class TaskStatus(Enum):
    PENDING    = 1
    IN_PROGRESS = 2
    DONE       = 3
    CANCELLED  = 4

HIGH_PRIORITY_THRESHOLD = 7

def process_task_good(task):
    if task["status"] == TaskStatus.IN_PROGRESS:
        if task["priority"] > HIGH_PRIORITY_THRESHOLD:
            return True

print("Anti-Pattern 2: Magic Numbers → Named constants and Enums")


# ═══════════════════════════════════════════════════════════════
# Anti-Pattern 3: MUTABLE DEFAULT ARGUMENTS
# ═══════════════════════════════════════════════════════════════

# ❌ BAD — mutable default is shared across all calls!
def append_to_list_bad(item, lst=[]):
    lst.append(item)
    return lst

print(append_to_list_bad(1))  # [1]
print(append_to_list_bad(2))  # [1, 2] ← BUG! shared state

# ✅ GOOD
def append_to_list_good(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(append_to_list_good(1))  # [1]
print(append_to_list_good(2))  # [2] ← correct


# ═══════════════════════════════════════════════════════════════
# Anti-Pattern 4: USING EXCEPTIONS FOR FLOW CONTROL
# ═══════════════════════════════════════════════════════════════

# ❌ BAD
def get_user_bad(users, id):
    try:
        return users[id]
    except KeyError:
        return None  # exceptions are slow, use .get()

# ✅ GOOD
def get_user_good(users, id):
    return users.get(id)  # returns None if not found, no exception

print("\nAnti-patterns demonstrated and corrected.")

---
## 6. Quiz <a id='quiz'></a>

Test your understanding before looking at the answers!

### Q1. What pattern enforces a single instance of a class?

<details>
<summary>▶ Reveal Answer</summary>

**Singleton Pattern.**

It ensures a class has only one instance and provides a global access point. In Python, the cleanest approaches are:
1. Using a metaclass that stores instances in `_instances` dict
2. Using a module-level instance (modules are imported once and cached by Python's import system)
3. Using `__new__` to return the existing instance if one exists

**Caution:** Singletons introduce global state, making unit testing harder. Always consider whether you truly need a Singleton or can use dependency injection instead.

</details>

---

### Q2. When would you use Strategy vs State pattern?

<details>
<summary>▶ Reveal Answer</summary>

**Strategy:** Algorithms are interchangeable and **the client chooses** which to use. The context doesn't change based on internal state — it just delegates to the selected algorithm. Example: choosing a sorting algorithm, a payment method, or a compression codec.

**State:** The object itself changes its behaviour **based on its own internal state**. Transitions happen automatically as the object progresses through its lifecycle. Example: a traffic light cycles through RED→GREEN→YELLOW→RED, or an order progresses through PENDING→PAID→SHIPPED→DELIVERED.

**Key difference:** In Strategy, the algorithm is injected from outside. In State, the object manages its own state transitions internally.

</details>

---

### Q3. How do Python's `@decorator` syntax and the Decorator design pattern relate?

<details>
<summary>▶ Reveal Answer</summary>

Python's `@decorator` syntax is a direct, idiomatic implementation of the **Decorator design pattern**. Both:
- Wrap an existing object/function with new behaviour
- Preserve the original interface (via `functools.wraps` for functions)
- Can be stacked (multiple decorators = multiple wrappers)
- Don't modify the original code

The difference is that the GoF Decorator is class-based (wrapping objects), while Python's `@decorator` is typically function-based. Python also supports class decorators that wrap classes, which is even closer to the original GoF intent.

```python
# This:
@timer
def my_func(): ...

# Is exactly equivalent to:
my_func = timer(my_func)
```

</details>

---
## 7. Interview Questions & Answers <a id='interview'></a>

---

### Q1. Implement the Observer pattern from scratch.

<details>
<summary>▶ Reveal Answer (with code)</summary>

See Section 3.1 above for a full implementation. Key points to mention in an interview:

1. **Subject** maintains a list of observers and notifies them on state changes
2. **Observers** implement a common interface (`update` method)
3. **Loose coupling** — Subject doesn't know the concrete type of its observers
4. **Python enhancement** — use `Protocol` instead of ABC for duck-typed observers
5. **Thread safety** — in production, use locks when modifying `_listeners`
6. **Memory leaks** — observers keep subjects alive; consider `weakref` for long-lived subjects

</details>

---

### Q2. Describe Factory vs Abstract Factory.

<details>
<summary>▶ Reveal Answer</summary>

**Factory Method:** A single method (or function) that creates one type of product. The subclass decides which concrete class to instantiate.
```
notification_factory("email") → EmailNotification
```

**Abstract Factory:** Creates **families of related objects** without specifying their concrete classes. It groups multiple factory methods that belong together.
```
# WindowsUIFactory creates: WindowsButton, WindowsCheckbox, WindowsDialog
# MacUIFactory creates:     MacButton,     MacCheckbox,     MacDialog
```

**Rule of thumb:**
- Factory Method → one product with variants
- Abstract Factory → multiple related products that must be used together (a "product family")

</details>

---

### Q3. How do Python decorators implement the Decorator pattern? Can you build a decorator factory?

</details>

In [ ]:
# ── Interview Q3: Decorator factory with configurable behaviour ───────────────
import functools
import logging
from typing import Optional

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)


def audit_log(
    action: str,
    log_args: bool = True,
    log_result: bool = False,
):
    """
    Decorator factory — configurable audit logging.

    Usage:
        @audit_log("user.login", log_args=True)
        def login(username, password): ...
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            call_info = f"action={action!r}"
            if log_args:
                call_info += f", args={args}, kwargs={kwargs}"
            log.info(f"[AUDIT] START {call_info}")
            result = func(*args, **kwargs)
            if log_result:
                log.info(f"[AUDIT] END   result={result!r}")
            else:
                log.info(f"[AUDIT] END   {action!r} completed")
            return result
        return wrapper
    return decorator


@audit_log("task.create", log_args=True, log_result=True)
def create_task(title: str, priority: int = 5) -> dict:
    return {"id": 99, "title": title, "priority": priority, "status": "pending"}


@audit_log("user.delete", log_args=False)
def delete_user(user_id: int) -> bool:
    return True


task = create_task("Review PR #42", priority=8)
print()
delete_user(user_id=17)

---
## 8. Problem Statement + Solution <a id='problem'></a>

### 🧩 Problem: Implement a Plugin System using Factory + Registry Pattern

**Requirements:**
1. Build a `PluginRegistry` that lets plugins self-register using a class decorator
2. Plugins implement a common `Plugin` interface with `name`, `version`, and `run(data)` method
3. The registry must support: `register`, `get`, `list_all`, `run_all`
4. Include a factory function that creates plugins by name
5. Plugins should handle errors gracefully and report status

**Bonus:** Add a pipeline that runs plugins in order and passes output to the next plugin

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Optional
import time


# ── Plugin Interface ──────────────────────────────────────────────────────────
@dataclass
class PluginResult:
    plugin_name: str
    success: bool
    data: Any
    error: Optional[str] = None
    duration_ms: float = 0.0

    def __repr__(self):
        status = "✓" if self.success else "✗"
        return (
            f"{status} [{self.plugin_name}] "
            f"{f'error: {self.error}' if not self.success else f'data keys: {list(self.data.keys()) if isinstance(self.data, dict) else type(self.data).__name__}'} "
            f"({self.duration_ms:.1f}ms)"
        )


class Plugin(ABC):
    name: str
    version: str = "1.0.0"

    @abstractmethod
    def run(self, data: dict) -> dict: ...

    def safe_run(self, data: dict) -> PluginResult:
        """Runs the plugin and wraps result in PluginResult."""
        start = time.perf_counter()
        try:
            result = self.run(data)
            return PluginResult(
                plugin_name=self.name,
                success=True,
                data=result,
                duration_ms=(time.perf_counter() - start) * 1000
            )
        except Exception as e:
            return PluginResult(
                plugin_name=self.name,
                success=False,
                data=data,     # pass through unchanged on failure
                error=str(e),
                duration_ms=(time.perf_counter() - start) * 1000
            )


# ── Plugin Registry ───────────────────────────────────────────────────────────
class PluginRegistry:
    _registry: dict[str, type[Plugin]] = {}

    @classmethod
    def register(cls, plugin_cls: type[Plugin]) -> type[Plugin]:
        """Class decorator — registers a plugin class."""
        if not hasattr(plugin_cls, 'name'):
            raise AttributeError(f"{plugin_cls.__name__} must define a 'name' attribute")
        cls._registry[plugin_cls.name] = plugin_cls
        print(f"[Registry] Registered plugin: {plugin_cls.name!r} v{plugin_cls.version}")
        return plugin_cls

    @classmethod
    def get(cls, name: str) -> type[Plugin]:
        if name not in cls._registry:
            available = list(cls._registry.keys())
            raise KeyError(f"Plugin {name!r} not found. Available: {available}")
        return cls._registry[name]

    @classmethod
    def list_all(cls) -> list[str]:
        return list(cls._registry.keys())

    @classmethod
    def create(cls, name: str, **kwargs) -> Plugin:
        """Factory method — creates a plugin instance by name."""
        return cls.get(name)(**kwargs)


# ── Pipeline ──────────────────────────────────────────────────────────────────
class PluginPipeline:
    """Runs plugins in sequence, passing each output as input to the next."""

    def __init__(self, *plugin_names: str):
        self._plugins: list[Plugin] = [
            PluginRegistry.create(name) for name in plugin_names
        ]

    def run(self, initial_data: dict) -> tuple[dict, list[PluginResult]]:
        data = initial_data.copy()
        results: list[PluginResult] = []

        for plugin in self._plugins:
            result = plugin.safe_run(data)
            results.append(result)
            if result.success:
                data = result.data  # pipe output → next input
            # on failure, data is unchanged (graceful degradation)

        return data, results


# ── Concrete Plugins ──────────────────────────────────────────────────────────
@PluginRegistry.register
class ValidationPlugin(Plugin):
    name = "validator"
    version = "1.2.0"

    REQUIRED_FIELDS = {"title", "user_id"}

    def run(self, data: dict) -> dict:
        missing = self.REQUIRED_FIELDS - set(data.keys())
        if missing:
            raise ValueError(f"Missing required fields: {missing}")
        data["validated"] = True
        return data


@PluginRegistry.register
class EnrichmentPlugin(Plugin):
    name = "enricher"
    version = "1.0.0"

    def run(self, data: dict) -> dict:
        import datetime
        return {
            **data,
            "created_at": datetime.datetime.now().isoformat(),
            "slug": data["title"].lower().replace(" ", "-"),
            "priority_label": "high" if data.get("priority", 5) >= 8 else "normal",
        }


@PluginRegistry.register
class PersistencePlugin(Plugin):
    name = "persister"
    version = "2.0.0"

    _store: list[dict] = []

    def run(self, data: dict) -> dict:
        import random
        data["id"] = random.randint(1000, 9999)
        self._store.append(data)
        print(f"    [Persister] Saved task #{data['id']}: {data['title']!r}")
        return data


# ── Run the pipeline ──────────────────────────────────────────────────────────
print(f"\nAll plugins: {PluginRegistry.list_all()}")

pipeline = PluginPipeline("validator", "enricher", "persister")

print("\n--- Pipeline Run 1: Valid data ---")
final, results = pipeline.run({
    "title": "Deploy to production",
    "user_id": 42,
    "priority": 9
})
for r in results:
    print(f"  {r}")
print(f"  Final keys: {list(final.keys())}")

print("\n--- Pipeline Run 2: Missing field (graceful failure) ---")
final2, results2 = pipeline.run({"title": "Orphan task"})  # missing user_id
for r in results2:
    print(f"  {r}")

---
## 9. Real-World Use-Case: Event-Driven Notification System <a id='usecase'></a>

### 🏗️ Architecture

We're building a **Task Manager notification system** that combines:

| Pattern | Role |
|---------|------|
| **Observer** | EventBus — tasks publish events, services subscribe |
| **Factory** | `NotificationFactory` — creates right channel (email/SMS/push) |
| **Strategy** | `DeliveryStrategy` — retry, immediate, batched |
| **Command** | `NotificationCommand` — undoable, queueable notification jobs |
| **Builder** | `NotificationBuilder` — fluent API to compose notifications |
| **Registry** | `ChannelRegistry` — self-registering delivery channels |

---

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  FULL IMPLEMENTATION: Event-Driven Notification System for a Task Manager
# ════════════════════════════════════════════════════════════════════════════

from __future__ import annotations
import time
import uuid
import datetime
import functools
from abc import ABC, abstractmethod
from collections import defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Any, Callable, Optional, Protocol, runtime_checkable


# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN MODELS
# ─────────────────────────────────────────────────────────────────────────────

class Priority(Enum):
    LOW    = 1
    MEDIUM = 5
    HIGH   = 8
    URGENT = 10

class TaskStatus(Enum):
    PENDING     = auto()
    IN_PROGRESS = auto()
    DONE        = auto()
    OVERDUE     = auto()

@dataclass
class User:
    id: int
    name: str
    email: str
    phone: str
    device_token: str
    notification_prefs: list[str] = field(default_factory=lambda: ["email", "push"])

@dataclass
class Task:
    id: int
    title: str
    assignee: User
    creator: User
    priority: Priority = Priority.MEDIUM
    status: TaskStatus = TaskStatus.PENDING
    due_date: Optional[datetime.datetime] = None


# ─────────────────────────────────────────────────────────────────────────────
# NOTIFICATION MODELS (Builder Pattern)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Notification:
    id: str
    channel: str
    recipient: str
    subject: str
    body: str
    priority: str = "normal"
    metadata: dict = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())


class NotificationBuilder:
    """Fluent builder for composing notifications."""

    def __init__(self):
        self._channel    = "email"
        self._recipient  = ""
        self._subject    = ""
        self._body       = ""
        self._priority   = "normal"
        self._metadata: dict = {}

    def via(self, channel: str)         -> NotificationBuilder: self._channel   = channel;   return self
    def to(self, recipient: str)        -> NotificationBuilder: self._recipient = recipient;  return self
    def subject(self, subject: str)     -> NotificationBuilder: self._subject   = subject;    return self
    def body(self, body: str)           -> NotificationBuilder: self._body      = body;       return self
    def urgent(self)                    -> NotificationBuilder: self._priority  = "urgent";   return self
    def with_meta(self, **kwargs)       -> NotificationBuilder: self._metadata.update(kwargs); return self

    def build(self) -> Notification:
        if not self._recipient or not self._body:
            raise ValueError("Notification must have a recipient and body")
        return Notification(
            id=str(uuid.uuid4())[:8],
            channel=self._channel,
            recipient=self._recipient,
            subject=self._subject,
            body=self._body,
            priority=self._priority,
            metadata=self._metadata,
        )


# ─────────────────────────────────────────────────────────────────────────────
# DELIVERY CHANNELS (Factory + Registry Pattern)
# ─────────────────────────────────────────────────────────────────────────────

class DeliveryResult:
    def __init__(self, success: bool, channel: str, recipient: str, message: str = ""):
        self.success   = success
        self.channel   = channel
        self.recipient = recipient
        self.message   = message

    def __repr__(self):
        icon = "✓" if self.success else "✗"
        return f"  {icon} [{self.channel.upper():<5}] → {self.recipient:<25} {self.message}"


class DeliveryChannel(ABC):
    channel_name: str

    @abstractmethod
    def deliver(self, notification: Notification) -> DeliveryResult: ...


class ChannelRegistry:
    """Registry + Factory for delivery channels."""
    _channels: dict[str, type[DeliveryChannel]] = {}

    @classmethod
    def register(cls, channel_cls: type[DeliveryChannel]) -> type[DeliveryChannel]:
        cls._channels[channel_cls.channel_name] = channel_cls
        return channel_cls

    @classmethod
    def create(cls, name: str) -> DeliveryChannel:
        if name not in cls._channels:
            raise KeyError(f"Channel {name!r} not registered. Available: {list(cls._channels)}")
        return cls._channels[name]()

    @classmethod
    def available(cls) -> list[str]: return list(cls._channels)


@ChannelRegistry.register
class EmailChannel(DeliveryChannel):
    channel_name = "email"
    def deliver(self, n: Notification) -> DeliveryResult:
        # Simulate SMTP call
        return DeliveryResult(True, "email", n.recipient, f"subject={n.subject!r}")

@ChannelRegistry.register
class SMSChannel(DeliveryChannel):
    channel_name = "sms"
    def deliver(self, n: Notification) -> DeliveryResult:
        body_preview = n.body[:40] + ("..." if len(n.body) > 40 else "")
        return DeliveryResult(True, "sms", n.recipient, f"msg={body_preview!r}")

@ChannelRegistry.register
class PushChannel(DeliveryChannel):
    channel_name = "push"
    def deliver(self, n: Notification) -> DeliveryResult:
        return DeliveryResult(True, "push", n.recipient, f"token={n.recipient[:12]}... title={n.subject!r}")

@ChannelRegistry.register
class SlackChannel(DeliveryChannel):
    channel_name = "slack"
    def deliver(self, n: Notification) -> DeliveryResult:
        return DeliveryResult(True, "slack", n.recipient, f"webhook_posted: {n.subject!r}")


# ─────────────────────────────────────────────────────────────────────────────
# DELIVERY STRATEGIES (Strategy Pattern)
# ─────────────────────────────────────────────────────────────────────────────

DeliveryStrategy = Callable[[list[Notification]], list[DeliveryResult]]

def immediate_delivery(notifications: list[Notification]) -> list[DeliveryResult]:
    """Strategy: deliver each notification immediately."""
    results = []
    for n in notifications:
        channel = ChannelRegistry.create(n.channel)
        results.append(channel.deliver(n))
    return results


def batched_delivery(notifications: list[Notification]) -> list[DeliveryResult]:
    """Strategy: group by channel, deliver in batches."""
    by_channel: dict[str, list[Notification]] = defaultdict(list)
    for n in notifications:
        by_channel[n.channel].append(n)

    results = []
    for channel_name, batch in by_channel.items():
        print(f"  [Batch] Delivering {len(batch)} {channel_name} notification(s)...")
        channel = ChannelRegistry.create(channel_name)
        for n in batch:
            results.append(channel.deliver(n))
    return results


# ─────────────────────────────────────────────────────────────────────────────
# NOTIFICATION SERVICE (Command + Observer)
# ─────────────────────────────────────────────────────────────────────────────

class NotificationService:
    """Core service — queues and delivers notifications using a chosen strategy."""

    def __init__(self, strategy: DeliveryStrategy = immediate_delivery):
        self._strategy = strategy
        self._queue: list[Notification] = []
        self._sent: list[Notification] = []
        self._failed: list[Notification] = []

    def set_strategy(self, strategy: DeliveryStrategy) -> None:
        self._strategy = strategy

    def queue(self, notification: Notification) -> None:
        self._queue.append(notification)

    def flush(self) -> list[DeliveryResult]:
        if not self._queue:
            return []
        batch = self._queue.copy()
        self._queue.clear()
        results = self._strategy(batch)
        for r, n in zip(results, batch):
            (self._sent if r.success else self._failed).append(n)
        return results

    def stats(self) -> dict:
        return {"queued": len(self._queue), "sent": len(self._sent), "failed": len(self._failed)}


# ─────────────────────────────────────────────────────────────────────────────
# EVENT BUS + TASK EVENT HANDLERS (Observer Pattern)
# ─────────────────────────────────────────────────────────────────────────────

class TaskEventBus:
    """Domain event bus for the task manager."""

    def __init__(self, notification_service: NotificationService):
        self._notif = notification_service
        self._handlers: dict[str, list[Callable]] = defaultdict(list)
        self._register_default_handlers()

    def _register_default_handlers(self):
        self.on("task.created",   self._on_task_created)
        self.on("task.assigned",  self._on_task_assigned)
        self.on("task.completed", self._on_task_completed)
        self.on("task.overdue",   self._on_task_overdue)
        self.on("task.commented", self._on_task_commented)

    def on(self, event: str, handler: Callable) -> None:
        self._handlers[event].append(handler)

    def emit(self, event: str, payload: dict) -> None:
        print(f"\n  ▶ EVENT: {event}")
        for handler in self._handlers[event]:
            handler(payload)

    # ── Event Handlers ────────────────────────────────────────────────────────

    def _on_task_created(self, p: dict):
        task: Task = p["task"]
        # Notify creator via their preferred channels
        for channel in task.creator.notification_prefs:
            recipient = task.creator.email if channel == "email" else (
                task.creator.phone if channel == "sms" else task.creator.device_token
            )
            notif = (
                NotificationBuilder()
                .via(channel)
                .to(recipient)
                .subject(f"Task created: {task.title}")
                .body(f"Hi {task.creator.name}, your task '{task.title}' has been created and assigned to {task.assignee.name}.")
                .with_meta(task_id=task.id, event="task.created")
                .build()
            )
            self._notif.queue(notif)

    def _on_task_assigned(self, p: dict):
        task: Task = p["task"]
        for channel in task.assignee.notification_prefs:
            recipient = task.assignee.email if channel == "email" else (
                task.assignee.phone if channel == "sms" else task.assignee.device_token
            )
            priority_flag = task.priority == Priority.URGENT
            builder = (
                NotificationBuilder()
                .via(channel)
                .to(recipient)
                .subject(f"[{'URGENT' if priority_flag else 'New'}] Task assigned: {task.title}")
                .body(f"Hi {task.assignee.name}, you've been assigned: '{task.title}' (Priority: {task.priority.name}).")
                .with_meta(task_id=task.id)
            )
            if priority_flag:
                builder = builder.urgent()
            self._notif.queue(builder.build())

    def _on_task_completed(self, p: dict):
        task: Task = p["task"]
        notif = (
            NotificationBuilder()
            .via("email")
            .to(task.creator.email)
            .subject(f"✓ Task completed: {task.title}")
            .body(f"{task.assignee.name} marked '{task.title}' as done.")
            .with_meta(task_id=task.id, completed_by=task.assignee.id)
            .build()
        )
        self._notif.queue(notif)

    def _on_task_overdue(self, p: dict):
        task: Task = p["task"]
        # Send to both assignee AND creator, across multiple channels
        for user in [task.assignee, task.creator]:
            for channel in ["email", "push"]:
                if channel not in user.notification_prefs:
                    continue
                recipient = user.email if channel == "email" else user.device_token
                notif = (
                    NotificationBuilder()
                    .via(channel)
                    .to(recipient)
                    .subject(f"⚠ OVERDUE: {task.title}")
                    .body(f"Task '{task.title}' was due on {task.due_date} and has not been completed.")
                    .urgent()
                    .with_meta(task_id=task.id, user_id=user.id)
                    .build()
                )
                self._notif.queue(notif)

    def _on_task_commented(self, p: dict):
        task: Task  = p["task"]
        comment: str = p["comment"]
        commenter: User = p["commenter"]
        # Notify the other party
        recipient_user = task.creator if commenter.id != task.creator.id else task.assignee
        notif = (
            NotificationBuilder()
            .via("push")
            .to(recipient_user.device_token)
            .subject(f"New comment on: {task.title}")
            .body(f"{commenter.name}: {comment[:80]}")
            .with_meta(task_id=task.id, commenter_id=commenter.id)
            .build()
        )
        self._notif.queue(notif)


# ─────────────────────────────────────────────────────────────────────────────
# DEMO — FULL SYSTEM IN ACTION
# ─────────────────────────────────────────────────────────────────────────────

print("═" * 65)
print(" TASK MANAGER — Event-Driven Notification System Demo")
print("═" * 65)
print(f"Registered channels: {ChannelRegistry.available()}")

# Setup
alice = User(1, "Alice",   "alice@corp.com",   "+1-555-0101", "device-token-alice-abc123", ["email", "push"])
bob   = User(2, "Bob",     "bob@corp.com",     "+1-555-0202", "device-token-bob-xyz789",  ["email", "sms", "push"])
carol = User(3, "Carol",   "carol@corp.com",   "+1-555-0303", "device-token-carol-def456", ["push"])

notif_service = NotificationService(strategy=immediate_delivery)
event_bus     = TaskEventBus(notif_service)

# ── Scenario 1: Alice creates an urgent task and assigns it to Bob ────────────
print("\n┌─ Scenario 1: Create urgent task ─────────────────────────────┐")
deploy_task = Task(
    id=101, title="Deploy v3.0 to production",
    assignee=bob, creator=alice,
    priority=Priority.URGENT,
    status=TaskStatus.PENDING,
    due_date=datetime.datetime.now() + datetime.timedelta(hours=2)
)
event_bus.emit("task.created",  {"task": deploy_task})
event_bus.emit("task.assigned", {"task": deploy_task})
print("\n  Flushing notification queue (immediate strategy):")
for r in notif_service.flush(): print(r)

# ── Scenario 2: Bob comments, Carol is assigned, task goes overdue ────────────
print("\n┌─ Scenario 2: Comment + overdue ───────────────────────────────┐")
review_task = Task(
    id=102, title="Review Q4 security audit",
    assignee=carol, creator=bob,
    priority=Priority.HIGH,
    status=TaskStatus.OVERDUE,
    due_date=datetime.datetime.now() - datetime.timedelta(days=1)
)
event_bus.emit("task.assigned",  {"task": review_task})
event_bus.emit("task.commented", {"task": review_task, "comment": "Please prioritise this — security team is waiting.", "commenter": bob})
event_bus.emit("task.overdue",   {"task": review_task})
print("\n  Switching to BATCHED delivery strategy...")
notif_service.set_strategy(batched_delivery)
for r in notif_service.flush(): print(r)

# ── Scenario 3: Task completed ────────────────────────────────────────────────
print("\n┌─ Scenario 3: Task completed ──────────────────────────────────┐")
deploy_task.status = TaskStatus.DONE
event_bus.emit("task.completed", {"task": deploy_task})
notif_service.set_strategy(immediate_delivery)
for r in notif_service.flush(): print(r)

# ── Final stats ───────────────────────────────────────────────────────────────
print("\n" + "═" * 65)
print(f" Notification Service Stats: {notif_service.stats()}")
print("═" * 65)

---
## 🏁 Day 31 Summary

| Pattern | Key Idea | Python Tool |
|---------|----------|-------------|
| Singleton | One instance, global access | Metaclass / module-level instance |
| Factory | Create objects by type at runtime | Dict registry + factory function |
| Builder | Fluent step-by-step construction | Method chaining, return `self` |
| Adapter | Make incompatible interfaces work | Wrapper class |
| Decorator | Add behaviour without modifying | `@functools.wraps`, `__enter__` |
| Proxy | Controlled access / caching | Wrapper with same interface |
| Observer | One-to-many event notification | `EventBus`, `Protocol` |
| Strategy | Swap algorithms at runtime | Callable / Protocol |
| Command | Encapsulate + undo operations | Command class + history stack |
| Registry | Self-registering plugins | Class decorator + dict |

### 📚 Further Reading
- *Design Patterns* — Gang of Four (original book)
- *Python Design Patterns* — Brandon Rhodes (pythonpatterns.com)
- *Fluent Python* Ch. 9-11 — Luciano Ramalho (decorator/descriptor deep dive)

---

### ➡️ Day 32: Dataclasses & Typing
Next we deep-dive into `@dataclass`, `frozen`, `__slots__`, `TypeVar`, `Generic`, `Protocol`, and runtime type checking.